In [1]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import pandas as pd
import math
import sys
from scipy.constants import g
from uncertainties import ufloat
import uncertainties.umath as unm
from uncertainties import unumpy as unp
from uncertainties.umath import sqrt
from uncertainties import ufloat
import sys
import os
percorso_main = os.path.abspath('..')
if percorso_main not in sys.path:
    sys.path.append(percorso_main)
from Libreria_python import analisi_dati as ad
from scipy.optimize import curve_fit

In [2]:
df=pd.read_excel('Urti.xlsx', sheet_name=0)
df=df.fillna('')
df


,Unnamed: 0,Carrello rosso (kg),Carrello blu (kg),massa 1 (kg),massa 2 (kg),Massa totale (kg),massa Libro 1 (kg),massa Libro 2 (kg),Errore sulla massa (kg)
0,Massa,,0.25452,0.25315,0.25375,0.76142,0.29007,0.18594,0.0005
1,Massa con magnete,0.25634,0.27538,,,0.78228,,,
2,Massa con molla,,0.25687,,,0.76324,,,


In [3]:
m_carrello=ufloat(df.iloc[0, 2], df.iloc[0, 8])
m_carrello_masse=ufloat(df.iloc[0, 5], df.iloc[0, 8])

#Seconda Legge di Newton: Piano Orizzontale


In [4]:
df1=pd.read_excel('Urti.xlsx', sheet_name=1)
df1=df1.fillna('')
df1

,Unnamed: 0,Interpolazione carrello,Unnamed: 2,Interpolazione carrello+masse,Unnamed: 4
0,,m (kg),sm (kg),m (kg),sm (kg)
1,1,0.249,0.04,0.769,0.043
2,2,0.253,0.043,0.755,0.038
3,3,0.247,0.042,0.749,0.038
4,x_best,0.249594,,0.756774,
5,sx_best,0.024023,,0.022787,


In [5]:
m1_interpolata=ufloat(df1.iloc[4, 1], df1.iloc[5, 1])
m2_interpolata=ufloat(df1.iloc[4, 3], df1.iloc[5, 3])

# Seconda Legge di Newton: Piano Inclinato

In [6]:
df2=pd.read_excel('Urti.xlsx', sheet_name=2)
df2=df2.fillna('')
df2

,Unnamed: 0,Interpolazione piano inclinato,Unnamed: 2,Unnamed: 3,Unnamed: 4,Angolo,incertezza
0,,m,sm,θ,sθ,10.0,1.0
1,1,0.264,0.002,0.176,0.002,,
2,2,0.257,0.002,0.16,0.003,,
3,3,0.26,0.003,0.175,0.0023,,
4,x_best,0.260409,,0.172425,,,
5,sx_best,0.001279,,0.001348,,,


In [7]:
m3_interpolata=ufloat(df2.iloc[4, 1], df2.iloc[5, 1])
theta_misurato=ufloat(np.radians(df2.iloc[0, 5]), np.radians(df2.iloc[0, 6]))
theta_interpolato=ufloat(df2.iloc[4, 3], df2.iloc[5, 3])

In [8]:
print("=" * 80)
ad.t_test(m1_interpolata, m_carrello, nome1="m1_interpolata", nome2="m_carrello");
ad.t_test(m2_interpolata, m_carrello_masse, nome1="m2_interpolata", nome2="m_carrello_masse");
ad.t_test(m3_interpolata, m_carrello, nome1="m3_interpolata", nome2="m_carrello");
ad.t_test(theta_interpolato, theta_misurato);
print("=" * 80)

Test di compatibilità tra m1_interpolata e m_carrello: |t| = 0.20499809
Test di compatibilità tra m2_interpolata e m_carrello_masse: |t| = 0.20383887
Test di compatibilità tra m3_interpolata e m_carrello: |t| = 4.2878102
Test di compatibilità tra misura 1 e misura 2: |t| = 0.12042043


# Teorema dell'Impulso


In [9]:
df3=pd.read_excel('Urti.xlsx', sheet_name=3)
df3=df3.fillna('')
df3

,Unnamed: 0,Massa (kg),Area (N*s),Incertezza (N*s),Velocità iniziale (m/s),Velocità finale (m/s),Impulso (N*s),ΔE (J),Coeff. di restituzione
0,Magnete,0.27538,-0.26,0.028868,0.56,-0.54,-0.302918,0.166605,0.964286
1,Molla,0.25687,-0.29,0.040415,0.64,-0.62,-0.323656,0.203903,0.968750


La correzione statistica (Distribuzione Uniforme)
L'istante in cui avviene il picco reale della forza è completamente casuale: può cadere esattamente su un punto di campionamento (errore geometrico zero) o esattamente a metà tra due punti (errore massimo). Tutti i punti intermedi hanno la stessa probabilità di verificarsi.
Quando un errore è distribuito uniformemente tra un valore minimo (0) e un valore massimo ($E_{max}$), la teoria degli errori (regolata dalle norme ISO/GUM sulla metrologia) impone di dividere l'errore massimo per $\sqrt{3}$ per ottenere la deviazione standard sperimentale.
Nel tuo caso, l'errore massimo di area che abbiamo calcolato è:

$$E_{max} = \frac{1}{4} |F_{max}| \delta t$$
La deviazione standard statistica ($\sigma_{stat}$), ovvero l'incertezza più realistica e non sovrastimata, diventa:

$$\sigma_{stat} = \frac{E_{max}}{\sqrt{3}} = \frac{\frac{1}{4} |F_{max}| \delta t}{\sqrt{3}} = \frac{1}{4\sqrt{3}} |F_{max}| \delta t \approx 0.144 \cdot |F_{max}| \delta t$$



In [ ]:
i_magnete_sperimentale=ufloat(df3.iloc[0, 2], df3.iloc[0, 3].round(3))
i_molla_sperimentale=ufloat(df3.iloc[1, 2], df3.iloc[1, 3].round(3))
i_magnete_teorico=df3.iloc[0, 6]
i_molla_teorico=df3.iloc[1, 6]
ad.compatibilita_valore(i_magnete_sperimentale, i_magnete_teorico, "Misura 1");
ad.compatibilita_valore(i_molla_sperimentale, i_molla_teorico, "Misura 2");

Compatibilità di Misura 1 con -0.302918: |t| = 1.479931
Compatibilità di Misura 2 con -0.3236562: |t| = 0.841405


# Urti tra Carrelli

In [10]:
df4=pd.read_excel('Urti.xlsx', sheet_name=4)
df4=df4.fillna('')
df4

,Unnamed: 0,Carrello rosso,Unnamed: 2,Unnamed: 3,Unnamed: 4,Carrello blu,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12
0,,Massa (kg),Velocità iniziale (m/s),Velocità finale (m/s),sv (m/s),Massa (kg),Velocità iniziale (m/s),Velocità finale (m/s),sv (m/s),Quantità di moto iniziale (kg*m/s),Quantità di moto finale (kg*m/s),Energia inizale (J),Energia finale (J)
1,Magnete,0.25634,0.501,-0.019,0.008,0.27538,0,0.485,,0.128426,0.128689,0.032171,0.032434
2,,0.25634,0.497,-0.229,,0.78228,0,0.243,,0.127401,0.131392,0.031659,0.029818
3,,0.76324,0.349,0.166,,0.27538,0,0.485,,0.266371,0.260257,0.046482,0.042904
4,Velcro,0.25634,0.468,0.221,0.024,0.27538,0,0.221,,0.119967,0.11751,0.028072,0.012985
5,,0.25634,0.563,0.132,0.012,0.78228,0,0.14,,0.144319,0.143356,0.040626,0.0099
6,,0.76324,0.425,0.344,0.015,0.27538,0,0.344,,0.324377,0.357285,0.06893,0.061453


In [11]:
m_R = unp.uarray(df4.iloc[1:4, 1].to_numpy(), 0.0005)
m_B = unp.uarray(df4.iloc[1:4, 4].to_numpy(), 0.0005)
v_i_R = unp.uarray(df4.iloc[1:4, 2].to_numpy(), 0.005)

vR_fM = unp.uarray(df4.iloc[1:4, 3].to_numpy(), 0.005)
vR_bM = unp.uarray(df4.iloc[1:4, 6].to_numpy(), 0.005)
v_fV = unp.uarray(df4.iloc[4:7, 3].to_numpy(), 0.005)

v_f_R = ((m_R - m_B) / (m_R + m_B)) * v_i_R
v_f_B = ((2 * m_R) / (m_R + m_B)) * v_i_R

print("=" * 70)

for i in range(len(m_R)):
    print(f"Set {i+1}: Velocità finale carrello rosso = {v_f_R[i]:.3P}, Velocità finale carrello blu = {v_f_B[i]:.3P}")
    ad.t_test(vR_fM[i], v_f_R[i])
    ad.t_test(vR_bM[i], v_f_B[i])
    print("=" * 70)

v_i_rv = unp.uarray(df4.iloc[4:7, 2].to_numpy(), 0.005)

v_f = (m_R / (m_R + m_B)) * v_i_rv

for i in range(len(m_B)):
    print(f"Set {i+1}: Velocità finale = {v_f[i]:.3P}")
    ad.t_test(v_fV[i], v_f[i])
    print("=" * 70)

ValueError: could not convert string to float: ''

# Attrito Statico


In [ ]:
df5=pd.read_excel('Urti.xlsx', sheet_name=5)
df5=df5.fillna('')
df5

,1 libro,Unnamed: 1,2 libri,Unnamed: 3
0,Massa (kg),F (N),Massa (kg),F (N)
1,0.29007,0.6812,0.47601,0.9359


In [ ]:
μ_s1=df5.iloc[1, 1]/(ufloat(df5.iloc[1, 0], 0.0005)*g)
μ_s2=df5.iloc[1, 3]/(ufloat(df5.iloc[1, 2], 0.0005)*g)
print("="*70)
print(f"Il valore misurato di μ_s1 è: {μ_s1:.4P}")
print(f"Il valore misurato di μ_s2 è: {μ_s2:.4P}")
print("="*70)

Il valore misurato di μ_s1 è: 0.2395±0.0004
Il valore misurato di μ_s2 è: 0.2005±0.0002


# Attrito Dinamico

In [ ]:
df6=pd.read_excel('Urti.xlsx', sheet_name=6)
df6=df6.fillna('')
df6

,Unnamed: 0,Coefficiente angolare
0,,1.478000
1,,1.519200
2,,1.511300
3,,1.517600
4,,1.524700
5,Media,1.510160
6,Errore della media,0.008319


In [ ]:
a=ufloat(df6.iloc[5, 1], df6.iloc[6,1])
theta = ufloat(np.radians(10), np.radians(1))
μ_d= unp.tan(theta)-a/(g*unp.cos(theta))
μ_d_atteso = 0.005                                  #l'ho torvato sul sito pasco come valore fornito a questi identici carrelli e piano inclinato
print("="*70)
print(f"Il valore misurato di μ_d è: {μ_d:.2P}")
ad.compatibilita_valore(μ_d, μ_d_atteso)
print("="*70)

Il valore misurato di μ_d è: 0.020±0.018
Compatibilità di misura con 0.005: |t| = 0.85298935
